In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import sys
sys.path.append("../../")

In [2]:
from MDP.EventContracts.EventContractsMDP import EventContractsMDP
from Query.EventContracts.EventContractQuery import EventContractQuery
from Query.EventContracts.EventContractValue import EventContractValue
from Query.EventContracts.EventContractStructure import EventContractStructure

# ── Step 1: Fetch the Pricer ──────────────────────────────────────────
mdp = EventContractsMDP(source="KALSHI")
pricer = mdp.get_pricer(
    {
        "ticker": "KXRATECUTCOUNT-26DEC31-T3",
        "timestamp": "live",
    }
)

# ── Step 2: Build the Query ──────────────────────────────────────────
q = EventContractQuery(
    structure=EventContractStructure.OUTRIGHT,
    value=EventContractValue.PRICE,
    ticker="KXRATECUTCOUNT-26DEC31-T3",
)

# ── Step 3: Resolve the Package ──────────────────────────────────────
package, risk_weights = q.resolve_package(pricer_or_curve=pricer)

# ── Step 4: Build the Value Map ──────────────────────────────────────
value_map = q.build_value_map(
    pricer_or_curve=pricer,
    package=package,
    risk_weights=risk_weights,
)

# ── Step 5: Evaluate ─────────────────────────────────────────────────
price = value_map.apply(EventContractValue.PRICE)
prob = value_map.apply(EventContractValue.PROBABILITY)
volume = value_map.apply(EventContractValue.VOLUME)
oi = value_map.apply(EventContractValue.OPEN_INTEREST)

print(f"Price:       {price}")
print(f"Probability: {prob}")
print(f"Volume:      {volume}")
print(f"OI:          {oi}")


quantity = 100_000

avg_price = value_map.apply(
    EventContractValue.MARKET_IMPACT_AVG_PRICE,
    quantity=quantity,
    side="yes",
)
slippage = value_map.apply(
    EventContractValue.MARKET_IMPACT_SLIPPAGE,
    quantity=quantity,
    side="yes",
)
total_cost = value_map.apply(
    EventContractValue.MARKET_IMPACT_TOTAL_COST,
    quantity=quantity,
    side="yes",
)

print(f"\n── Market Impact (buy 500 YES) ──")
print(f"Avg Fill:    {avg_price}")
print(f"Slippage:    {slippage}")
print(f"Total Cost:  ${total_cost:.2f}")

Price:       0.105
Probability: 0.105
Volume:      552.0
OI:          134981.0

── Market Impact (buy 500 YES) ──
Avg Fill:    0.19165369999999998
Slippage:    0.0816537
Total Cost:  $19165.37


In [3]:
pricer.get_orderbook()

{'yes':    price  quantity
 0  0.001    1043.0
 1  0.002     300.0
 2  0.011      13.0
 3  0.018    1124.0
 4  0.070    2000.0
 5  0.090    2350.0
 6  0.100    2066.0
 7  0.101      51.0,
 'no':     price  quantity
 0    0.01    4900.0
 1    0.02    2900.0
 2    0.03    1000.0
 3    0.25      39.0
 4    0.26      37.0
 ..    ...       ...
 63   0.85   25000.0
 64   0.86      45.0
 65   0.87    2033.0
 66   0.88       3.0
 67   0.89    3821.0
 
 [68 rows x 2 columns]}

In [4]:
mdp = EventContractsMDP(source="POLYMARKET")
pricer = mdp.get_pricer({
    "token_id": "113379839734351069617987084078322474966003108854908079701423911002443710490196",
})
pricer.latest_price()

0.27